In [ ]:
!pip install skl2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 22.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

# Estadística clásica (para calcular neutral_temp)
import statsmodels.formula.api as smf

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

# ONNX
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

In [ ]:
# 1. Descargar datos

url_meta = "https://github.com/CenterForTheBuiltEnvironment/ashrae-db-II/raw/master/v2.1.0/db_metadata.csv"
url_measurements = "https://github.com/CenterForTheBuiltEnvironment/ashrae-db-II/raw/master/v2.1.0/db_measurements_v2.1.0.csv.gz"

df_meta = pd.read_csv(url_meta)
df_measurements = pd.read_csv(url_measurements)

/tmp/ipykernel_18465/1846102160.py:9: DtypeWarning: Columns (5,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  df_measurements = pd.read_csv(url_measurements)


In [ ]:
# 2. Filtrar datos necesarios

df_acm = df_measurements.loc[
    (~df_measurements["ta"].isna())
    & (~df_measurements["thermal_sensation"].isna())
    & (~df_measurements["rh"].isna())
    & (~(df_measurements["t_out_isd"].isna()) | ~(df_measurements["t_out"].isna()))
].copy()

# Combinar temperatura exterior
df_acm["t_out_combined"] = df_acm["t_out_isd"].fillna(df_acm["t_out"])
df_acm = df_acm.drop(columns=["t_out_isd", "t_out"])

# Merge metadata
df_acm = df_acm.merge(
    df_meta[["building_id", "region", "building_type", "cooling_type", "records"]],
    on="building_id",
    how="left",
)

# Solo oficinas
df_acm = df_acm[df_acm["building_type"] == "office"]
df_acm = df_acm.drop(columns=["building_type"])

In [ ]:
# 3. Calcular temperatura neutral por edificio


def run_lm(bldg):
    try:
        lm_result = smf.ols(formula="ta ~ thermal_sensation", data=bldg).fit()
        if lm_result.pvalues["Intercept"] < 0.05:
            return lm_result.params["Intercept"]
        else:
            return np.nan
    except:
        return np.nan


df_models = df_acm.groupby("building_id").apply(run_lm).reset_index()
df_models.columns = ["building_id", "neutral_temp"]

# Agregar metadata
df_models = df_models.merge(
    df_meta[["building_id", "records", "cooling_type", "region"]],
    on="building_id",
    how="left",
)

# Calcular temperatura exterior promedio
df_models["t_out_mean"] = df_acm.groupby("building_id")["t_out_combined"].mean().values

# Eliminar NaN
df_models = df_models.dropna()

print("Total edificios usados:", len(df_models))

Total edificios usados: 300


/tmp/ipykernel_18465/663560779.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_models = df_acm.groupby('building_id').apply(run_lm).reset_index()


In [ ]:
# 4. Preparar dataset ML

# 'sparse' pasa a ser 'sparse_output'
encoder = OneHotEncoder(sparse_output=False)
cooling_encoded = encoder.fit_transform(df_models[["cooling_type"]])

# Features finales
X_numeric = df_models[["t_out_mean"]].values
X = np.hstack([X_numeric, cooling_encoded]).astype(np.float32)

y = df_models["neutral_temp"].values.astype(np.float32)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 5. Entrenar modelo ML

model = RandomForestRegressor(
    n_estimators=100, max_depth=None, min_samples_split=5, random_state=42
)

model.fit(X_train, y_train)

# Evaluación
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 1.7957383115385812
R2: 0.4942168197282947


In [ ]:
# 6. Exportar a ONNX

initial_type = [("float_input", FloatTensorType([None, X.shape[1]]))]

onnx_model = convert_sklearn(model, initial_types=initial_type, target_opset=10)

with open("adaptive_comfort_rf.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Modelo exportado como adaptive_comfort_rf.onnx")

Modelo exportado como adaptive_comfort_rf.onnx


In [ ]:
import numpy as np

# Nuevo ejemplo
t_out_nuevo = 28.6417
cooling_nuevo = "air conditioned"

# 1. Convertir temperatura exterior
X_num = np.array([[t_out_nuevo]])

# 2. One-hot encoding
X_cat = encoder.transform([[cooling_nuevo]])

# 3. Unir variables
X_final = np.hstack([X_num, X_cat]).astype(np.float32)

# 4. Predecir
pred = model.predict(X_final)

print("Temperatura neutral predicha:", pred[0])

Temperatura neutral predicha: 25.15589395904541


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
# Tomar un edificio real
ejemplo = df_models.iloc[0]

print("Valor real:", ejemplo["neutral_temp"])

# Preparar input
X_num = np.array([[ejemplo["t_out_mean"]]])
X_cat = encoder.transform([[ejemplo["cooling_type"]]])
X_final = np.hstack([X_num, X_cat]).astype(np.float32)

# Predecir
pred = model.predict(X_final)

print("Predicción:", pred[0])
print("Error:", abs(pred[0] - ejemplo["neutral_temp"]))

Valor real: 22.585738264421188
Predicción: 22.86554984162345
Error: 0.27981157720226335


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
nuevos = pd.DataFrame(
    {
        "t_out_mean": [15.29, 9.04, 28.85],
        "cooling_type": ["mixed mode", "air conditioned", "air conditioned"],
    }
)

X_num = nuevos[["t_out_mean"]].values
X_cat = encoder.transform(nuevos[["cooling_type"]])

X_final = np.hstack([X_num, X_cat]).astype(np.float32)

preds = model.predict(X_final)

print(preds)

[22.86554984 21.38049512 25.95327348]
